# LSTM & GRU under the hood: watch the gates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudanum/MIA5150/blob/main/notebooks/sequential-models-rnn-gru-lstm/03_lstm_gru_gates.ipynb)

*MIA5150 — Generative AI & Large Language Models · University of Ottawa · Instructor: Imran Ahmad*

Notebook 2 showed *that* vanilla RNNs forget: gradients shrink exponentially as they travel back through time, so a signal from far in the past might as well not exist. LSTMs and GRUs are the classic fix — but they are usually used as black boxes (`nn.LSTM`, magic happens, loss goes down). In this notebook we open the box. You will re-implement both cells from scratch, prove line-for-line that your code is identical to PyTorch's, and then do something most practitioners never do: **watch the gates do their job** on a task that forces a network to carry one bit of information across 30 timesteps of noise — and discover where the textbook "write → carry → read" cartoon matches a trained network, and where reality is richer.

**Learning goals**

- Write the LSTM and GRU update equations and connect every term to a *mechanism* (write / keep / read).
- Implement an LSTM and a GRU cell from scratch in PyTorch — and **verify them against `nn.LSTMCell` / `nn.GRUCell`** to floating-point precision.
- Explain why a same-sized LSTM has exactly 4/3 the parameters of a GRU, and why the *forget-bias trick* is what makes long carries learnable at all.
- Read gate-activation plots: find the CARRY pattern a trained LSTM develops, understand why the WRITE looks different from the cartoon, and explain why the READ needs no output gate.

## 1 · The LSTM, equation by equation

A vanilla RNN keeps one state vector and rewrites it completely at every step: $h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$. The problem (notebook 2): whatever was in $h_{t-1}$ gets squashed through a $\tanh$ and mixed with the new input, so information decays fast.

The **LSTM** (Hochreiter & Schmidhuber, 1997) adds a second state — the **cell state** $c_t$ — that is never squashed and never matrix-multiplied. It only ever gets *scaled* and *added to*:

$$
\begin{aligned}
f_t &= \sigma(W_f x_t + U_f h_{t-1} + b_f) && \text{forget gate}\\
i_t &= \sigma(W_i x_t + U_i h_{t-1} + b_i) && \text{input gate}\\
o_t &= \sigma(W_o x_t + U_o h_{t-1} + b_o) && \text{output gate}\\
\tilde{c}_t &= \tanh(W_c x_t + U_c h_{t-1} + b_c) && \text{candidate content}\\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t && \text{cell update}\\
h_t &= o_t \odot \tanh(c_t) && \text{visible output}
\end{aligned}
$$

Three things to hold onto:

- **The cell state is a conveyor belt.** Look at the update $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$: old content is multiplied by a number near 1 (kept) or near 0 (erased), and new content is *added*. There is no $\tanh$ and no weight matrix on the path $c_{t-1} \to c_t$, so the local gradient $\partial c_t / \partial c_{t-1} = f_t$ can stay near 1 for hundreds of steps. That is the whole trick: an almost-additive highway through time.
- **Gates are valves, computed from the here-and-now.** Each gate is a sigmoid of the current input and the previous output, so each of its values lives in $(0, 1)$: 0 = closed, 1 = open. The network *learns when* to write ($i_t$), when to keep ($f_t$), and when to read ($o_t$) — per unit, per timestep.
- **$h_t$ is a peek at the cell, not the memory itself.** The output gate decides how much of the ($\tanh$-squashed) cell the rest of the network gets to see this step.

A note on notation: above, each gate has its own weight matrices for clarity. PyTorch (and our code below) packs the four gates into **one** big matrix multiply for efficiency — same math, one fused operation. PyTorch's packed order is $i, f, \tilde{c}, o$.

## 2 · The GRU: same idea, two gates, no separate cell

The **GRU** (Cho et al., 2014) drops the separate cell state: the hidden state *is* the memory, and a single **update gate** $z_t$ interpolates between the old state and a fresh candidate:

$$
\begin{aligned}
r_t &= \sigma(W_{ir} x_t + b_{ir} + W_{hr} h_{t-1} + b_{hr}) && \text{reset gate}\\
z_t &= \sigma(W_{iz} x_t + b_{iz} + W_{hz} h_{t-1} + b_{hz}) && \text{update gate}\\
n_t &= \tanh\!\big(W_{in} x_t + b_{in} + r_t \odot (W_{hn} h_{t-1} + b_{hn})\big) && \text{candidate}\\
h_t &= (1 - z_t) \odot n_t + z_t \odot h_{t-1} && \text{state update}
\end{aligned}
$$

- **Update gate $z_t$ — the interpolation lever.** Because $z_t$ multiplies the *old* state, **$z_t \approx 1$ means "keep what I have, ignore the candidate"** (carry) and **$z_t \approx 0$ means "overwrite with the candidate"** (write). One gate does the work of the LSTM's forget + input pair: think of it as the constraint $f_t = 1 - i_t$ made literal.
- **Reset gate $r_t$ — sits inside the candidate.** It controls how much of the old state the candidate is allowed to see. With $r_t \approx 0$ the candidate is computed (almost) as if there were no past — useful when the past has just become irrelevant.
- **No output gate, no $\tanh$ on readout.** $h_t$ is exposed directly. A GRU cannot "hold a secret" the way an LSTM cell can; in exchange it uses fewer parameters (we count them exactly in section 5).

> **Convention warning (read once, saves a day).** The equations above are **PyTorch's exact convention**, and two details are easy to get wrong when copying from a paper or blog:
> 1. The reset gate multiplies the *already-transformed* hidden state: $r_t \odot (W_{hn} h_{t-1} + b_{hn})$, **not** $W_{hn}(r_t \odot h_{t-1})$. The original GRU paper does the latter; PyTorch does the former. They differ whenever $b_{hn} \neq 0$.
> 2. $z_t$ sits on the **old** state: $h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$. Some tutorials write the interpolation the other way around, with the gate on the *new* value — the same model up to $z \leftrightarrow 1 - z$, but mix the two and your verification against `nn.GRUCell` *will* fail and your gate plots will read upside-down. In everything below, **high $z$ = keep the old state**.

In [ ]:
# ============================================================
# Setup — torch / numpy / matplotlib are preinstalled in Colab,
# so there is nothing to pip-install for this notebook.
# ============================================================
import math
import random
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Deterministic runs: same seeds -> same plots for everyone in class
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
print(f"torch {torch.__version__}")

In [ ]:
# ============================================================
# Config — the knobs. Tweak and re-run (Runtime -> Run all).
# ============================================================
INPUT_SIZE  = 1      # one scalar per timestep
HIDDEN_SIZE = 16     # LSTM/GRU state size
T           = 30     # sequence length = how far the flag must be carried

GATE_BIAS   = 2.0    # start forget/update gate OPEN, write/reveal gates shut
                     # (the forget-bias trick — see section 6). Try 0.0 and
                     # watch training fail: vanishing gradients, live.

STEPS       = 400    # Adam steps for each base model (a few hundred is plenty)
DISTRACTOR_STEPS = 600   # the distractor variant is a little harder
N_DISTRACTORS = 2
BATCH_SIZE  = 128
LR          = 1e-2

## 3 · LSTM from scratch — and proof it's the real thing

Below is a complete LSTM cell. It keeps the four gates packed into one pair of weight matrices exactly like `nn.LSTMCell` (gate order $i, f, \tilde{c}, o$), and its `forward` returns a **dict of all gate activations** so we can inspect them later. Reading tip: the whole cell is one matrix multiply, one `chunk`, four nonlinearities, and two elementwise lines.

In [ ]:
class LSTMCellScratch(nn.Module):
    """A single LSTM timestep, with explicit packed weights.

    Shapes (d = input_size, h = hidden_size):
        weight_ih: (4h, d)   weight_hh: (4h, h)   biases: (4h,)
    Gate order inside the packed dimension: i, f, g, o  (PyTorch's order).
    """

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(4 * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size))
        self.bias_ih   = nn.Parameter(torch.empty(4 * hidden_size))
        self.bias_hh   = nn.Parameter(torch.empty(4 * hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        # Same default init as nn.LSTMCell: U(-1/sqrt(h), +1/sqrt(h))
        bound = 1.0 / math.sqrt(self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -bound, bound)

    def forward(self, x, state):
        h_prev, c_prev = state
        # One fused multiply for all four gates, then split them back out
        gates = (x @ self.weight_ih.T + self.bias_ih
                 + h_prev @ self.weight_hh.T + self.bias_hh)
        i_pre, f_pre, g_pre, o_pre = gates.chunk(4, dim=1)

        i_t = torch.sigmoid(i_pre)  # input gate:  how much candidate to WRITE
        f_t = torch.sigmoid(f_pre)  # forget gate: how much old cell to KEEP
        g_t = torch.tanh(g_pre)     # candidate content c~ (the new stuff)
        o_t = torch.sigmoid(o_pre)  # output gate: how much cell to REVEAL

        c_t = f_t * c_prev + i_t * g_t   # conveyor belt: scale the old, add the new
        h_t = o_t * torch.tanh(c_t)      # peek at the cell through the output gate

        return {"h": h_t, "c": c_t, "i": i_t, "f": f_t, "g": g_t, "o": o_t}

In [ ]:
# ------------------------------------------------------------
# Verification: our cell vs nn.LSTMCell, identical weights & inputs
# ------------------------------------------------------------
torch.manual_seed(SEED)
ref  = nn.LSTMCell(INPUT_SIZE, HIDDEN_SIZE)          # the PyTorch original
mine = LSTMCellScratch(INPUT_SIZE, HIDDEN_SIZE)

# Copy every parameter across, so any difference is *math*, not weights
with torch.no_grad():
    mine.weight_ih.copy_(ref.weight_ih)
    mine.weight_hh.copy_(ref.weight_hh)
    mine.bias_ih.copy_(ref.bias_ih)
    mine.bias_hh.copy_(ref.bias_hh)

x  = torch.randn(8, INPUT_SIZE)
h0 = torch.randn(8, HIDDEN_SIZE)
c0 = torch.randn(8, HIDDEN_SIZE)

ref_h, ref_c = ref(x, (h0, c0))
out = mine(x, (h0, c0))

torch.testing.assert_close(out["h"], ref_h)
torch.testing.assert_close(out["c"], ref_c)

print(f"max |delta h| = {(out['h'] - ref_h).abs().max():.2e}")
print(f"max |delta c| = {(out['c'] - ref_c).abs().max():.2e}")
print("assert_close passed: the from-scratch LSTM is exactly the real thing.")

## 4 · GRU from scratch — mind the convention

Same exercise for the GRU. The two lines that must match PyTorch *exactly* (both flagged in section 2): the reset gate multiplies the transformed state, $r_t \odot (W_{hn} h_{t-1} + b_{hn})$, and the update is $h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$ — $z$ on the old state. PyTorch packs the three gates in the order $r, z, n$.

In [ ]:
class GRUCellScratch(nn.Module):
    """A single GRU timestep, with explicit packed weights.

    Gate order inside the packed dimension: r, z, n  (PyTorch's order).
    """

    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.weight_ih = nn.Parameter(torch.empty(3 * hidden_size, input_size))
        self.weight_hh = nn.Parameter(torch.empty(3 * hidden_size, hidden_size))
        self.bias_ih   = nn.Parameter(torch.empty(3 * hidden_size))
        self.bias_hh   = nn.Parameter(torch.empty(3 * hidden_size))
        self.reset_parameters()

    def reset_parameters(self):
        bound = 1.0 / math.sqrt(self.hidden_size)
        for p in self.parameters():
            nn.init.uniform_(p, -bound, bound)

    def forward(self, x, h_prev):
        gates_x = x @ self.weight_ih.T + self.bias_ih
        gates_h = h_prev @ self.weight_hh.T + self.bias_hh
        x_r, x_z, x_n = gates_x.chunk(3, dim=1)
        h_r, h_z, h_n = gates_h.chunk(3, dim=1)

        r_t = torch.sigmoid(x_r + h_r)        # reset gate
        z_t = torch.sigmoid(x_z + h_z)        # update gate (sits on the OLD state)
        n_t = torch.tanh(x_n + r_t * h_n)     # candidate: reset applied to the
                                              # *transformed* hidden state (PyTorch)
        h_t = (1 - z_t) * n_t + z_t * h_prev  # z ~ 1: keep old | z ~ 0: write new

        return {"h": h_t, "r": r_t, "z": z_t, "n": n_t}

In [ ]:
torch.manual_seed(SEED)
ref  = nn.GRUCell(INPUT_SIZE, HIDDEN_SIZE)
mine = GRUCellScratch(INPUT_SIZE, HIDDEN_SIZE)

with torch.no_grad():
    mine.weight_ih.copy_(ref.weight_ih)
    mine.weight_hh.copy_(ref.weight_hh)
    mine.bias_ih.copy_(ref.bias_ih)
    mine.bias_hh.copy_(ref.bias_hh)

x  = torch.randn(8, INPUT_SIZE)
h0 = torch.randn(8, HIDDEN_SIZE)

ref_h = ref(x, h0)
out   = mine(x, h0)

torch.testing.assert_close(out["h"], ref_h)
print(f"max |delta h| = {(out['h'] - ref_h).abs().max():.2e}")
print("assert_close passed: the from-scratch GRU matches nn.GRUCell exactly.")
print("(If this ever fails in your own experiments, suspect the two convention")
print(" details from section 2 first: r on the transformed state, z on the old state.)")

## 5 · Counting parameters: why LSTM = (4/3) × GRU

Both cells compute $G$ gates, each with its own slice of the packed weights. For input size $d$, hidden size $h$, and PyTorch's two bias vectors, one "gate's worth" of parameters costs

$$
\underbrace{h \cdot d}_{W_{ih}\ \text{slice}} \;+\; \underbrace{h \cdot h}_{W_{hh}\ \text{slice}} \;+\; \underbrace{h + h}_{b_{ih} + b_{hh}} \;=\; h\,(d + h + 2)
$$

The LSTM has $G = 4$ ($i, f, \tilde{c}, o$); the GRU has $G = 3$ ($r, z, n$). So

$$
\frac{\#\text{LSTM}}{\#\text{GRU}} = \frac{4\,h\,(d+h+2)}{3\,h\,(d+h+2)} = \frac{4}{3}
$$

— **exactly**, biases included, for any $d$ and $h$. The GRU's 25% discount is the price of giving up the separate cell state and the output gate. The cell below checks the arithmetic against the real modules.

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())

d, h = INPUT_SIZE, HIDDEN_SIZE

per_gate  = h * (d + h + 2)      # one gate: h*d (W_ih) + h*h (W_hh) + 2 biases of h
lstm_pred = 4 * per_gate
gru_pred  = 3 * per_gate

lstm_real = count_params(nn.LSTMCell(d, h))
gru_real  = count_params(nn.GRUCell(d, h))

print(f"input d = {d}, hidden h = {h}")
print(f"per gate: h*(d+h+2) = {h}*({d}+{h}+2) = {per_gate}")
print(f"LSTM: predicted {lstm_pred}, actual {lstm_real}")
print(f"GRU : predicted {gru_pred}, actual {gru_real}")
print(f"ratio LSTM/GRU = {lstm_real / gru_real:.6f}   (4/3 = {4/3:.6f})")

assert lstm_pred == lstm_real and gru_pred == gru_real
assert math.isclose(lstm_real / gru_real, 4 / 3)
print("arithmetic confirmed: LSTM carries exactly 4/3 the GRU's parameters.")

## 6 · The carry task: a laboratory for gates

To *see* gates do their job we need a task where carrying information is forced. The **carry task**:

- Sequences of length $T = 30$, one scalar per step.
- At $t = 0$ the input is a **flag**: $+1$ or $-1$, chosen at random per sequence.
- Every other timestep is pure noise, $x_t \sim U(-0.5, 0.5)$.
- **Target: the flag**, read out from the *final* hidden state (MSE regression).

The model gets exactly one glimpse of the answer, at $t = 0$, and must reproduce it after 29 steps of noise. There is no shortcut through the inputs — the only way to solve the task is to *write* the flag into memory, *carry* it across the sequence, and *read* it out at the end. (Same family as the adding problem from notebook 2, stripped down until the mechanism is fully visible.)

A sanity baseline: always predicting $0$ costs $\mathrm{MSE} = \mathbb{E}[\text{flag}^2] = 1$. Anything well below that means real carrying is happening.

**One implementation detail — and it is a famous one.** We initialize the forget gate's bias *positive* and the input/output biases *negative* (`GATE_BIAS` in the config): the **forget-bias trick** (Gers, Schmidhuber & Cummins, 2000), generalized to all three gates. Why it matters: with the default init every gate starts near $0.5$, so a freshly written value decays by $\approx 0.5$ per step, and the gradient that would teach the network to carry dies long before it reaches the write — notebook 2's vanishing gradient, on day one of training. Starting with the carry lane *open* gives learning a fighting chance. (Exercise 3 asks you to set `GATE_BIAS = 0` and watch the loss stick at the baseline.)

In [ ]:
# Fresh random batch every training step: no fixed dataset to overfit.
def make_carry_batch(batch_size, seq_len=T):
    """x: (batch, seq_len, 1) — flag at t=0, noise elsewhere.  y: (batch, 1)."""
    x = torch.rand(batch_size, seq_len, 1) - 0.5                 # U(-0.5, 0.5)
    flags = torch.randint(0, 2, (batch_size, 1)).float() * 2 - 1  # {-1, +1}
    x[:, 0, 0] = flags[:, 0]
    return x, flags

x_demo, y_demo = make_carry_batch(4)
print("x shape:", tuple(x_demo.shape), " y shape:", tuple(y_demo.shape))
print("first sequence, first 6 inputs:", x_demo[0, :6, 0].tolist(), " ... target:", y_demo[0, 0].item())
print("flag values in batch:", sorted(set(y_demo[:, 0].tolist())))

In [ ]:
class CarryLSTM(nn.Module):
    """The scratch LSTM cell looped over time + a linear readout on the final h."""

    def __init__(self, hidden_size):
        super().__init__()
        self.cell = LSTMCellScratch(INPUT_SIZE, hidden_size)
        self.head = nn.Linear(hidden_size, 1)
        # The forget-bias trick, generalized: start with the carry lane OPEN
        # (f ~ 0.9) and the write/reveal gates mostly shut. Without this, the
        # long carry never gets off the ground (Try-it exercise 3 ablates it).
        with torch.no_grad():
            h = hidden_size                 # packed gate order: i, f, g, o
            self.cell.bias_ih[h:2*h] += GATE_BIAS   # forget gate: start open
            self.cell.bias_ih[0:h]   -= GATE_BIAS   # input gate: start shut
            self.cell.bias_ih[3*h:]  -= GATE_BIAS   # output gate: start shut

    def forward(self, x, collect=False):
        batch, seq_len, _ = x.shape
        h = x.new_zeros(batch, self.cell.hidden_size)
        c = x.new_zeros(batch, self.cell.hidden_size)
        trace = defaultdict(list)
        for t in range(seq_len):
            out = self.cell(x[:, t, :], (h, c))
            h, c = out["h"], out["c"]
            if collect:
                for key in ("i", "f", "o", "c"):
                    trace[key].append(out[key])
                trace["yhat"].append(self.head(h))
        y = self.head(h)
        if collect:  # (batch, seq_len, hidden) per entry
            trace = {k: torch.stack(v, dim=1) for k, v in trace.items()}
            return y, trace
        return y


torch.manual_seed(SEED)          # reproducible init *and* reproducible batches
lstm = CarryLSTM(HIDDEN_SIZE)
opt = torch.optim.Adam(lstm.parameters(), lr=LR)

losses = []
for step in range(STEPS):
    x, y = make_carry_batch(BATCH_SIZE)
    loss = F.mse_loss(lstm(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 100 == 0 or step == STEPS - 1:
        print(f"step {step:4d} | train MSE {loss.item():.4f}")

# Honest evaluation: fresh batches the model has never seen
with torch.no_grad():
    eval_mse = float(np.mean([
        F.mse_loss(lstm(xb), yb).item()
        for xb, yb in (make_carry_batch(256) for _ in range(50))
    ]))
print(f"\neval MSE on fresh batches: {eval_mse:.4f}   (baseline 'always 0' = 1.0)")
print("the carry is solved." if eval_mse < 0.05 else "not fully solved yet — try more STEPS.")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(losses, lw=1)
ax.axhline(1.0, ls="--", c="gray", label="baseline: always predict 0 (MSE = 1)")
ax.set_yscale("log")
ax.set_xlabel("Adam step")
ax.set_ylabel("train MSE (log scale)")
ax.set_title("Carry task — training the scratch LSTM")
ax.legend()
plt.show()

**What to notice.** The loss sits near the baseline for the first stretch — the model is opening the carry lane — and then falls orders of magnitude below it. Two honest observations: the from-scratch cell really trains end to end, and nothing about the architecture *forces* the carry — the equations only *allow* it. Gradient descent discovers the strategy. Next we catch it in the act.

## 7 · Watching the gates: the cartoon, and the reality

Now the payoff. We take the **trained** LSTM, feed it one sequence with flag $+1$, and record every gate at every timestep. Six panels:

1. the **input** the model sees,
2–4. the three **gates** $f_t, i_t, o_t$ (bold = mean over all 16 units, faint = individual units),
5. the **cell state** $c_t$ of the few units that end up carrying the flag,
6. the **readout** $\hat{y}_t$ — what the model would predict if we stopped and read at step $t$.

The textbook cartoon predicts: input gate **spikes at $t = 0$ and shuts** (WRITE), forget gate **near 1 through the middle** (CARRY), output gate **rises near the end** (READ). Run the cell, then let's compare carefully — **two of these panels will match the cartoon beautifully, and two will surprise you.** The surprises are where the real understanding is.

In [ ]:
# One fixed example: flag = +1, reproducible noise
gen = torch.Generator().manual_seed(SEED + 1)
x_ex = torch.rand(1, T, 1, generator=gen) - 0.5
x_ex[0, 0, 0] = 1.0

lstm.eval()
with torch.no_grad():
    y_ex, tr = lstm(x_ex, collect=True)

ts    = np.arange(T)
sig   = x_ex[0, :, 0].numpy()
gates = {k: tr[k][0].numpy() for k in ("i", "f", "o")}  # (T, hidden) each
cell  = tr["c"][0].numpy()                              # (T, hidden)
yhat  = tr["yhat"][0, :, 0].numpy()                     # (T,)

def plot_gate(ax, values, color, label):
    ax.plot(ts, values, color=color, alpha=0.15, lw=0.8)       # every unit, faint
    ax.plot(ts, values.mean(axis=1), color=color, lw=2.5, label="mean over units")
    ax.axvline(0, color="k", lw=0.5, alpha=0.3)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel(label)
    ax.legend(loc="center right", fontsize=8)

# The units that actually carry: largest |cell state| at the final step
top_units = np.argsort(np.abs(cell[-1]))[-3:][::-1]

fig, axes = plt.subplots(6, 1, figsize=(10, 15), sharex=True)

axes[0].plot(ts, sig, lw=1.5, color="black")
axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set_ylabel("input $x_t$")
axes[0].set_title("One carried flag (+1) through the trained LSTM")

plot_gate(axes[1], gates["f"], "tab:red",    "forget $f_t$")
plot_gate(axes[2], gates["i"], "tab:green",  "input $i_t$")
plot_gate(axes[3], gates["o"], "tab:orange", "output $o_t$")

for u in top_units:
    axes[4].plot(ts, cell[:, u], lw=1.8, label=f"unit {u}")
axes[4].axhline(0, color="gray", lw=0.5)
axes[4].set_ylabel("cell $c_t$")
axes[4].legend(loc="best", fontsize=8)

axes[5].plot(ts, yhat, lw=2, color="tab:blue")
axes[5].axhline(1.0, ls="--", color="gray", label="true flag = +1")
axes[5].axhline(0, color="gray", lw=0.5)
axes[5].set_ylabel(r"readout $\hat{y}_t$")
axes[5].set_xlabel("timestep t")
axes[5].legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

print(f"final readout: y_hat = {yhat[-1]:+.3f}  (true flag = +1)")

**Reading the six panels** — what matches the cartoon, and what doesn't:

- **Input.** A single $+1$ spike at $t = 0$ in a sea of small noise. After that, the inputs carry *no* information about the target.
- **Forget gate (red) — textbook-perfect CARRY.** The units doing the carrying pin $f_t$ very close to 1 for the whole sequence: $f_t \approx 1$ means $c_t \approx c_{t-1}$, the conveyor belt rolling. This is the panel to hold next to notebook 2's dying RNN gradients.
- **Input gate (green) — here reality departs from the cartoon.** $i_t$ opens at $t = 0$… and stays *partly* open. Why? Because the flag is the only **persistent** component of the input: zero-mean noise integrates away to nothing, but a $+1$ flag keeps adding $+1$-ish contributions. The network has turned its cell into a **matched filter** — an integrator — which is a strictly *better* solution than write-once-then-shut on this task. The first write still matters most: look at the cell state jump to the correct sign within the first couple of steps.
- **Cell state — the flag, persisting.** The carrying units ramp in the flag's direction and **never cross zero**: $\mathrm{sign}(c_t)$ *is* the flag, held for 30 steps. The magnitude grows (that's the integrator at work; $\tanh(c)$ saturates, which is harmless). Also notice: only a few of the 16 units bother — the rest idle near zero. **Specialization**, free of charge.
- **Output gate (orange) — flat, and the equations say it must be.** "Rises near the end" cannot happen here, for a deep reason: the loss only listens at $t = T-1$, and the gates see only $(x_t, h_{t-1})$ — the model has **no clock**, nothing that distinguishes step 29 from step 15. There is no gradient asking $o$ to change over time, so it doesn't. The READ is simply delegated to the head, which reads the final $h$ directly. (Real tasks give the model a clock — an end-of-sequence marker, a time input; exercise 4 asks you to build one.)
- **Readout.** $\hat{y}_t$ locks onto $+1$ within the first several steps and stays there: once $\tanh(c)$ saturates, the head has a rock-steady signal.

So: the WRITE happened (up front, then gently), the CARRY is exactly as advertised, and the READ needs no gate at all. The cartoon is a special case — trained gates are analog, and they learn whatever the loss rewards.

## 8 · The GRU solves it too — one lever instead of three

Same task, same training, scratch GRU (with the update-gate bias starting in "keep" mode, the GRU's share of the forget-bias trick). What should we expect from the equations of section 2?

- **Write at the start:** with $h_{-1} = 0$, the first update is just $h_0 = (1 - z_0) \odot n_0$ — the candidate gets in through the $(1 - z)$ fraction. Expect $z$ to sit slightly *below* its cruising level early on, while $h$ is still empty.
- **Carry through the middle:** $z_t$ high. In PyTorch's convention $z$ sits on the *old* state — **high $z$ = keep, low $z$ = overwrite** — and one lever does the work of forget + input.
- **Candidate $n_t$:** free to keep "voting" for the stored flag through $h_{t-1}$ (the GRU's version of the self-refresh we just saw in the LSTM).
- **Reset gate $r_t$:** mostly a bystander on this task — when $z$ keeps the candidate's influence small, it hardly matters how much of the past that candidate was looking at.
- **Read:** no output gate exists — $h_t$ is always fully visible, so the head can read the memory the whole way.

In [ ]:
class CarryGRU(nn.Module):
    """The scratch GRU cell looped over time + a linear readout on the final h."""

    def __init__(self, hidden_size):
        super().__init__()
        self.cell = GRUCellScratch(INPUT_SIZE, hidden_size)
        self.head = nn.Linear(hidden_size, 1)
        # GRU share of the forget-bias trick: the update gate starts in
        # "keep the old state" mode (z ~ 0.9). Packed gate order: r, z, n.
        with torch.no_grad():
            h = hidden_size
            self.cell.bias_ih[h:2*h] += GATE_BIAS

    def forward(self, x, collect=False):
        batch, seq_len, _ = x.shape
        h = x.new_zeros(batch, self.cell.hidden_size)
        trace = defaultdict(list)
        for t in range(seq_len):
            out = self.cell(x[:, t, :], h)
            h = out["h"]
            if collect:
                for key in ("r", "z", "n", "h"):
                    trace[key].append(out[key])
                trace["yhat"].append(self.head(h))
        y = self.head(h)
        if collect:
            trace = {k: torch.stack(v, dim=1) for k, v in trace.items()}
            return y, trace
        return y


torch.manual_seed(SEED)
gru = CarryGRU(HIDDEN_SIZE)
opt = torch.optim.Adam(gru.parameters(), lr=LR)

gru_losses = []
for step in range(STEPS):
    x, y = make_carry_batch(BATCH_SIZE)
    loss = F.mse_loss(gru(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    gru_losses.append(loss.item())
    if step % 100 == 0 or step == STEPS - 1:
        print(f"step {step:4d} | train MSE {loss.item():.4f}")

with torch.no_grad():
    gru_eval = float(np.mean([
        F.mse_loss(gru(xb), yb).item()
        for xb, yb in (make_carry_batch(256) for _ in range(50))
    ]))
print(f"\nGRU eval MSE on fresh batches: {gru_eval:.4f}   (baseline 'always 0' = 1.0)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(gru_losses, lw=1, color="tab:purple")
ax.axhline(1.0, ls="--", c="gray", label="baseline: always predict 0 (MSE = 1)")
ax.set_yscale("log")
ax.set_xlabel("Adam step")
ax.set_ylabel("train MSE (log scale)")
ax.set_title("Carry task — training the scratch GRU")
ax.legend()
plt.show()

In [ ]:
gen = torch.Generator().manual_seed(SEED + 1)   # the SAME example sequence as before
x_ex = torch.rand(1, T, 1, generator=gen) - 0.5
x_ex[0, 0, 0] = 1.0

gru.eval()
with torch.no_grad():
    y_ex_g, trg = gru(x_ex, collect=True)

zg    = trg["z"][0].numpy()      # (T, hidden)
rg    = trg["r"][0].numpy()
ng    = trg["n"][0].numpy()
hg    = trg["h"][0].numpy()
yhatg = trg["yhat"][0, :, 0].numpy()

top_units_g = np.argsort(np.abs(hg[-1]))[-3:][::-1]

fig, axes = plt.subplots(6, 1, figsize=(10, 15), sharex=True)

axes[0].plot(ts, sig, lw=1.5, color="black")
axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set_ylabel("input $x_t$")
axes[0].set_title("The same flag (+1) through the trained GRU")

plot_gate(axes[1], zg, "tab:red",   "update $z_t$")
plot_gate(axes[2], rg, "tab:green", "reset $r_t$")

axes[3].plot(ts, ng, color="tab:purple", alpha=0.15, lw=0.8)
axes[3].plot(ts, ng.mean(axis=1), color="tab:purple", lw=2.5)
axes[3].axhline(0, color="gray", lw=0.5)
axes[3].set_ylim(-1.05, 1.05)
axes[3].set_ylabel("candidate $n_t$")

for u in top_units_g:
    axes[4].plot(ts, hg[:, u], lw=1.8, label=f"unit {u}")
axes[4].axhline(0, color="gray", lw=0.5)
axes[4].set_ylabel("hidden $h_t$")
axes[4].legend(loc="best", fontsize=8)

axes[5].plot(ts, yhatg, lw=2, color="tab:blue")
axes[5].axhline(1.0, ls="--", color="gray", label="true flag = +1")
axes[5].axhline(0, color="gray", lw=0.5)
axes[5].set_ylabel(r"readout $\hat{y}_t$")
axes[5].set_xlabel("timestep t")
axes[5].legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

print(f"final readout: y_hat = {yhatg[-1]:+.3f}  (true flag = +1)")

**Reading the GRU panels** — the same protocol, one lever instead of three:

- **Update gate $z_t$ (red).** High the whole way — $z$ cruises around $0.8$ and above: mostly-keep, with a steady trickle of candidate. Notice that the GRU's write is **gradual**: each step admits only $(1 - z_t) \approx 15\%$ of the candidate, but because the candidate keeps voting the stored direction, $h$ still ramps to its final value over the opening steps. With $h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$, high $z$ means *keep the old state*; a dip toward 0 would mean *overwrite*. If you remember one line from this notebook: **the update gate is glued to the old state** (and if a tutorial shows you the opposite pattern, it has $z$ and $1 - z$ swapped — same model, flipped labels).
- **Reset gate $r_t$ (green).** Wanders without a strong pattern, exactly as predicted: while $z$ stays high the candidate barely influences $h$, so it hardly matters how much of the past that candidate was seeing. On tasks where the past periodically *becomes* irrelevant, $r$ is the gate that hides it — not needed here, where the whole point is to never forget.
- **Candidate $n_t$ (purple).** Keeps voting in the flag's direction mid-sequence — the GRU's self-refresh. Through $h_{t-1}$ the candidate can read what is stored and re-emit it, topping up whatever the $(1 - z)$ leak lets through.
- **Hidden state $h_t$ — the carry, directly visible.** The carrying units ramp to a definite signed value over the opening steps and then hold it, essentially flat, all the way to the end. No separate cell, no output gate: $h$ *is* the memory, and you are looking at it.
- **Readout.** Tracks $+1$ the whole way (nothing is hidden from the head) and finishes within a hair of the true flag.

Two architectures, one solution concept: **write early, keep the state nearly frozen, read at the end.** The LSTM does it with a private cell and three gates; the GRU with a single interpolation lever and 3/4 of the parameters.

## 9 · Making the WRITE visible: distractor flags

On the plain carry task, mid-sequence writes are *harmless* (the noise integrates away), so the input gate never needs to snap decisively — that is why the WRITE panel looked gentle. Change one rule and everything changes:

> Extra $\pm 1$ **distractor flags** land at random timesteps, and only the **last** flag in the sequence counts.

Now mid-sequence writes are *consequential*: integrating blindly would mix flags of opposite sign. The model must **notice flag moments, overwrite the cell, and hold between flags** — much closer to the textbook cartoon. Below we retrain on this variant and feed the trained model one sequence with flags $+1$ at $t=0$, $-1$ at $t=10$, $+1$ at $t=22$ (target: $+1$, the last one).

In [ ]:
def make_distractor_batch(batch_size, seq_len=T, n_distractors=N_DISTRACTORS):
    """Like make_carry_batch, plus extra ±1 flags at random later times.
    Target: the LAST flag in the sequence (earlier ones must be overwritten)."""
    x = torch.rand(batch_size, seq_len, 1) - 0.5
    x[:, 0, 0] = torch.randint(0, 2, (batch_size,)).float() * 2 - 1
    last_flag = x[:, 0, 0].clone()
    for b in range(batch_size):
        pos, _ = torch.sort(torch.randint(seq_len // 3, seq_len - 2, (n_distractors,)))
        signs = torch.randint(0, 2, (n_distractors,)).float() * 2 - 1
        x[b, pos, 0] = signs
        last_flag[b] = signs[-1]        # pos is sorted: signs[-1] is the latest flag
    return x, last_flag.unsqueeze(1)


torch.manual_seed(SEED)
lstm_dis = CarryLSTM(HIDDEN_SIZE)
opt = torch.optim.Adam(lstm_dis.parameters(), lr=LR)

dis_losses = []
for step in range(DISTRACTOR_STEPS):
    x, y = make_distractor_batch(BATCH_SIZE)
    loss = F.mse_loss(lstm_dis(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    dis_losses.append(loss.item())
    if step % 150 == 0 or step == DISTRACTOR_STEPS - 1:
        print(f"step {step:4d} | train MSE {loss.item():.4f}")

with torch.no_grad():
    dis_eval = float(np.mean([
        F.mse_loss(lstm_dis(xb), yb).item()
        for xb, yb in (make_distractor_batch(256) for _ in range(50))
    ]))
print(f"distractor-task eval MSE: {dis_eval:.4f}   (baseline 'always 0' = 1.0)")

# ---- one staged example: flags +1 @ t=0, -1 @ t=10, +1 @ t=22  (target: +1)
gen = torch.Generator().manual_seed(SEED + 2)
x_dis = torch.rand(1, T, 1, generator=gen) - 0.5
flag_times  = [0, 10, 22]
flag_values = [+1.0, -1.0, +1.0]
for tt, vv in zip(flag_times, flag_values):
    x_dis[0, tt, 0] = vv

lstm_dis.eval()
with torch.no_grad():
    y_dis, trd = lstm_dis(x_dis, collect=True)

i_dis  = trd["i"][0].numpy()
c_dis  = trd["c"][0].numpy()
y_dis_ = trd["yhat"][0, :, 0].numpy()
sig_dis = x_dis[0, :, 0].numpy()
top_d = np.argsort(np.abs(c_dis[-1]))[-3:][::-1]

fig, axes = plt.subplots(4, 1, figsize=(10, 11), sharex=True)

axes[0].plot(ts, sig_dis, lw=1.5, color="black")
axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set_ylabel("input $x_t$")
axes[0].set_title("Distractor task: flags +1, -1, +1 at t = 0, 10, 22 (target: the last one)")

plot_gate(axes[1], i_dis, "tab:green", "input $i_t$")

for u in top_d:
    axes[2].plot(ts, c_dis[:, u], lw=1.8, label=f"unit {u}")
axes[2].axhline(0, color="gray", lw=0.5)
axes[2].set_ylabel("cell $c_t$")
axes[2].legend(loc="best", fontsize=8)

axes[3].plot(ts, y_dis_, lw=2, color="tab:blue")
axes[3].axhline(1.0, ls="--", color="gray", label="target = last flag = +1")
axes[3].axhline(0, color="gray", lw=0.5)
axes[3].set_ylabel(r"readout $\hat{y}_t$")
axes[3].set_xlabel("timestep t")
axes[3].legend(loc="best", fontsize=8)

for ax in axes:                      # mark the flag moments on every panel
    for tt in flag_times[1:]:
        ax.axvline(tt, color="tab:red", ls=":", lw=1)

plt.tight_layout()
plt.show()

print(f"final readout: y_hat = {y_dis_[-1]:+.3f}  (target = last flag = +1)")

**What to notice.**

- **The readout is the star.** $\hat{y}_t$ settles near $+1$ after the first flag, **flips negative right after the distractor at $t = 10$**, and **flips back after $t = 22$**. The model re-reads the world at each flag and updates its stored belief on the fly — discrete writes, on cue. There is no way to produce this trace without writing at exactly those moments.
- **Input gate (green) — selective, and sign-specialized.** Watch the faint traces at the red dotted flag lines: in this trained model, individual units open their input gate *at specific flags* — one unit opens almost exclusively at the $-1$ distractor ($t = 10$), while others open at the $+1$ flags ($t = 0$ and $t = 22$). Writes are now expensive, so writing has become selective — and the work is split by sign. (Your exact cast of units may differ by seed; the selectivity won't.)
- **Cell state.** The carrying units change course at each flag — look for the kinks at $t = 10$ and $t = 22$ — and run nearly straight between flags while $f \approx 1$ keeps the belt rolling. On average the forget gate also dips *at* flag moments: erase, then write — the cartoon's WRITE, finally on stage.
- **Still analog, not digital.** The pattern is softer than the cartoon — trained gates always are — but the causal chain is now fully visible: **flags → gate openings → cell overwrites → readout flips.**

Optional extra: retrain with `N_DISTRACTORS = 4` or more. The more often the world changes, the sharper the write/select machinery becomes.

## Try it yourself

1. **Longer carry.** Set `T = 60` and retrain both models (you may want a few more `STEPS`). Does the forget gate still stay near 1 the whole way? Which architecture struggles first? For a real challenge, combine with distractors.
2. **Cell-state specialization.** After training, count how many of the 16 cell units actually carry signal on the example sequence (e.g. $|c_{T-1}| > 0.5$). Now retrain with `HIDDEN_SIZE = 4`. Does it still solve the task? What is the smallest hidden size that works — and what happens to the gate plots as you shrink it?
3. **Ablate the forget-bias trick.** Set `GATE_BIAS = 0.0` and re-run the whole notebook. The loss now sticks at the baseline — why? Connect this to notebook 2: with every gate starting near $0.5$, what happens to (a) the carried value and (b) the gradient reaching the write, across 29 steps? Why does opening the forget gate at init fix *both*?
4. **Give the model a clock.** In section 7 the output gate stayed flat because nothing distinguishes $t = 29$ from $t = 15$. Fix that: add a second input channel that ramps linearly from 0 to 1 over the sequence (or appends a $+1$ marker at the final step). Predict what happens to $o_t$, then run it. (For the full cartoon, also add a small penalty on $|\hat{y}_t|$ for $t < T - 1$ — "stay silent until the end" — and watch the output gate finally rise at the end.)

## Recap

- An LSTM adds a **cell state** that is only ever scaled (forget gate) and added to (input gate) — an additive highway that keeps gradients alive over long distances. The output gate decides when to reveal it.
- A GRU fuses forget + input into a single **update gate** $z_t$ that interpolates between old state and candidate: in PyTorch's convention $h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$, so **high $z$ = keep the old state**. The reset gate gates the candidate's view of the past.
- Both scratch cells match `nn.LSTMCell` / `nn.GRUCell` to `assert_close` precision — there is no magic inside the black box, just these equations.
- Same input/hidden size, an LSTM carries exactly **4/3×** a GRU's parameters (4 gates vs 3, biases included).
- The **forget-bias trick** (starting the carry lane open) is not a nicety — without it, the carry task is unlearnable in practice: the gradient dies before it reaches the write.
- Trained gates are **analog, and shaped by the task**: on the plain carry task the network becomes an integrator (write early, keep $f \approx 1$, let the head read); add distractors and the WRITE becomes selective and visible. The one cartoon element that never appears — "the output gate rises at the end" — needs something the task doesn't provide: a clock.

Next notebook: we swap the toy task for real text and race RNN vs GRU vs LSTM on IMDb sentiment.